In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
from dall_e import map_pixels, unmap_pixels, load_model
from dall_e import Encoder, Decoder
import torch

import torch.nn.functional as F

import onnxruntime as ort
import numpy as np


In [2]:
device = torch.device("cpu")
enc: Encoder = load_model("https://cdn.openai.com/dall-e/encoder.pkl", device)
# dec: Decoder = load_model("https://cdn.openai.com/dall-e/decoder.pkl", device)

In [3]:
enc.eval()
for param in enc.parameters():
    param.requires_grad = False

dummy_img = torch.randn(1, 3, 480, 640, device=device)
# dummy_img = torch.randn(1, 3, 120, 160, device="cuda")
torch.onnx.export(
    enc,
    dummy_img,
    "./enc.onnx",
    export_params=True,
    opset_version=13,
    input_names=["pixels"],
    # dynamic_axes={"pixels": {2: "height", 3: "width"}},
    dynamic_axes=None
)
print("ONNX model saved: enc.onnx")

/home/nvidia/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/nvidia/.local/lib/python3.10/site-packages/dall_e/encoder.py:88: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if x.shape[1] != self.input_channels:


ONNX model saved: enc.onnx


In [ ]:
# trtexec --onnx=enc.onnx --saveEngine=enc.trt --fp16

In [ ]:
trtexec \
  --onnx=./enc.onnx \
  --saveEngine=./enc.trt \
  --fp16 \
  --workspace=256 \
  --useSpinWait \
  --builderOptimizationLevel=0 \
  --minShapes=pixels:1x3x120x160 \
  --optShapes=pixels:1x3x240x320 \
  --maxShapes=pixels:1x3x240x320

In [ ]:
[05/03/2026-17:25:35] [I] Throughput: 5200.7 qps
[05/03/2026-17:25:35] [I] Latency: min = 0.195068 ms, max = 0.385681 ms, mean = 0.204975 ms, median = 0.204468 ms, percentile(90%) = 0.209106 ms, percentile(95%) = 0.211182 ms, percentile(99%) = 0.216583 ms
[05/03/2026-17:25:35] [I] Enqueue Time: min = 0.106934 ms, max = 0.43927 ms, mean = 0.124905 ms, median = 0.127197 ms, percentile(90%) = 0.135742 ms, percentile(95%) = 0.138611 ms, percentile(99%) = 0.153076 ms
[05/03/2026-17:25:35] [I] H2D Latency: min = 0.00476074 ms, max = 0.0446777 ms, mean = 0.00983314 ms, median = 0.0090332 ms, percentile(90%) = 0.0134277 ms, percentile(95%) = 0.0147705 ms, percentile(99%) = 0.0180664 ms
[05/03/2026-17:25:35] [I] GPU Compute Time: min = 0.182617 ms, max = 0.368713 ms, mean = 0.187595 ms, median = 0.1875 ms, percentile(90%) = 0.189514 ms, percentile(95%) = 0.190308 ms, percentile(99%) = 0.192383 ms
[05/03/2026-17:25:35] [I] D2H Latency: min = 0.00488281 ms, max = 0.0107422 ms, mean = 0.00754818 ms, median = 0.00756836 ms, percentile(90%) = 0.00897217 ms, percentile(95%) = 0.00927734 ms, percentile(99%) = 0.00976562 ms
[05/03/2026-17:25:35] [I] Total Host Walltime: 3.00056 s
[05/03/2026-17:25:35] [I] Total GPU Compute Time: 2.92741 s

In [ ]:
[05/03/2026-17:30:40] [I] === Performance summary ===
[05/03/2026-17:30:40] [I] Throughput: 5432.36 qps
[05/03/2026-17:30:40] [I] Latency: min = 0.187744 ms, max = 0.243408 ms, mean = 0.193056 ms, median = 0.192871 ms, percentile(90%) = 0.19458 ms, percentile(95%) = 0.19516 ms, percentile(99%) = 0.198303 ms
[05/03/2026-17:30:40] [I] Enqueue Time: min = 0.107178 ms, max = 0.207214 ms, mean = 0.118099 ms, median = 0.117676 ms, percentile(90%) = 0.123657 ms, percentile(95%) = 0.127563 ms, percentile(99%) = 0.139526 ms
[05/03/2026-17:30:40] [I] H2D Latency: min = 0.00488281 ms, max = 0.0474854 ms, mean = 0.00742402 ms, median = 0.00732422 ms, percentile(90%) = 0.00775146 ms, percentile(95%) = 0.00793457 ms, percentile(99%) = 0.0125732 ms
[05/03/2026-17:30:40] [I] GPU Compute Time: min = 0.178223 ms, max = 0.233154 ms, mean = 0.181983 ms, median = 0.181885 ms, percentile(90%) = 0.18335 ms, percentile(95%) = 0.183838 ms, percentile(99%) = 0.184631 ms
[05/03/2026-17:30:40] [I] D2H Latency: min = 0.00268555 ms, max = 0.00537109 ms, mean = 0.0036503 ms, median = 0.00366211 ms, percentile(90%) = 0.00427246 ms, percentile(95%) = 0.00439453 ms, percentile(99%) = 0.00476074 ms
[05/03/2026-17:30:40] [I] Total Host Walltime: 3.00054 s
[05/03/2026-17:30:40] [I] Total GPU Compute Time: 2.96632 s

In [ ]:
dec_cpu = dec.cpu().eval()

z = torch.randint(0, dec_cpu.vocab_size, size=(1, 30, 40))
dummy_codewords = F.one_hot(z, num_classes=dec_cpu.vocab_size).permute(0, 3, 1, 2).float()
torch.onnx.export(
    dec_cpu,
    dummy_codewords,
    "./dec.onnx",
    export_params=True,
    input_names=["codewords"],
    dynamic_axes={"codewords": {0: "batch", 2: "height", 3: "width"}},
)
print("ONNX model saved: dec.onnx")

In [ ]:
dummy_codewords.shape, dummy_codewords.dtype

In [ ]:
dec_cpu(dummy_codewords)

In [ ]:
trtexec \
  --onnx=./dec.onnx \
  --saveEngine=./dec.trt \
  --fp16 \
  --workspace=4096 \
  --useSpinWait
